# Quantum double-slit wave inversion

Regenerate the physical matrix and the inverse phases at the original grid and width.

1. Set up local FlagQuantum execution and plot rendering.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.cfd.double_slit_experiment.quantum_double_slit_experiment.overview import workflow
from examples.synthesis.applications.cfd.double_slit_experiment.quantum_double_slit_experiment.overview.main import print_report, save_animation, save_figures
from examples.synthesis.applications.cfd.double_slit_experiment.quantum_double_slit_experiment.overview.model import DEGREE, GRID, SOURCES

2. Retain the original spatial resolution, constants, and wave sources.

In [ ]:
print('Grid:', GRID, GRID)
print('Sources:', SOURCES)
assert GRID**2 == 256

3. Solve the absorbing-boundary matrix and run the full native inverse.

In [ ]:
result = workflow.run()
assert tuple(result.model.matrix.shape) == (256, 256)
print('Wave block normalization:', result.model.alpha)

4. Render the reference and postselected wave fields with the source phase normalization.

In [ ]:
figures = Path('notebook-figures')
save_figures(result, figures)
display(Image(filename=str(figures / 'wave-fields.png')))

5. Check the mapped Chebyshev reciprocal at the original degree.

In [ ]:
assert DEGREE == 191 and result.inverse.coefficients.numel() == 96
display(Image(filename=str(figures / 'inverse-poly.png')))

6. Regenerate all phases and compare with the independent singular-value response.

In [ ]:
assert result.inverse.phases.phases.numel() == 192
assert result.inverse.expected_fidelity >= 0.95
assert result.spectral_error < 1e-10

7. Verify full-state phase, postselection and the physical width constraint.

In [ ]:
assert result.resources.n_qubits <= 18
assert result.full_error < 1e-10 and result.work_probability == 0
print_report(result)

8. Animate the real component over a complete physical phase cycle.

In [ ]:
save_animation(result, figures)
display(Image(filename=str(figures / 'wave-cycle.gif'), format='gif'))